In [1]:
%%writefile common.py
"""
Shared utilities for the DQN / A2C / PPO CartPole-v1 assignment.
"""
import os
import csv
import random
import numpy as np
import torch
import gymnasium as gym


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_env(env_id: str, seed: int = 0):
    """Single (non-vectorized) environment, used by DQN and by evaluation."""
    env = gym.make(env_id)
    env.reset(seed=seed)
    env.action_space.seed(seed)
    return env


def make_vector_env(env_id: str, n_envs: int, seed: int = 0):
    """Synchronous vector environment used by A2C and PPO for parallel rollouts."""
    def make(rank):
        def _thunk():
            env = gym.make(env_id)
            env.reset(seed=seed + rank)
            return env
        return _thunk
    return gym.vector.SyncVectorEnv([make(i) for i in range(n_envs)])


class CSVLogger:
    """Simple append-only CSV logger, one row per call to .log()."""
    def __init__(self, path: str, fieldnames):
        self.path = path
        self.fieldnames = fieldnames
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(self.path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()

    def log(self, row: dict):
        with open(self.path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self.fieldnames)
            writer.writerow(row)


@torch.no_grad()
def evaluate_policy(env_id, action_fn, n_episodes=20, seed=0, device="cpu"):
    """
    Runs n_episodes on a single (non-vectorized, non-exploring) environment.
    action_fn(obs) -> int action. action_fn should already be deterministic/greedy;
    this function adds no exploration noise, matching the "no exploration during
    evaluation" requirement in the assignment.
    """
    env = gym.make(env_id)
    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + 10_000 + ep)
        done = False
        ep_return = 0.0
        while not done:
            action = action_fn(obs)
            obs, reward, terminated, truncated, _ = env.step(action)
            ep_return += reward
            done = terminated or truncated
        returns.append(ep_return)
    env.close()
    return float(np.mean(returns)), float(np.std(returns))


def mlp(input_dim, output_dim, hidden_size=128, n_hidden=2):
    """2-hidden-layer, 128-unit, ReLU MLP as specified in the reference setup."""
    layers = []
    last = input_dim
    for _ in range(n_hidden):
        layers += [torch.nn.Linear(last, hidden_size), torch.nn.ReLU()]
        last = hidden_size
    layers += [torch.nn.Linear(last, output_dim)]
    return torch.nn.Sequential(*layers)


Writing common.py


In [2]:
%%writefile dqn.py
"""
DQN on CartPole-v1.

Normal run:
    python dqn.py --seed 0 --out-dir results/dqn_normal

Ablation (no target network -> online network bootstraps itself):
    python dqn.py --seed 0 --no-target-network --out-dir results/dqn_no_target
"""
import argparse
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from common import set_seed, make_env, CSVLogger, evaluate_policy, mlp

ENV_ID = "CartPole-v1"


class QNetwork(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden_size=128):
        super().__init__()
        self.net = mlp(obs_dim, n_actions, hidden_size=hidden_size, n_hidden=2)

    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    """Fixed-size, preallocated numpy replay buffer."""
    def __init__(self, capacity, obs_dim):
        self.capacity = capacity
        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros((capacity,), dtype=np.int64)
        self.rewards = np.zeros((capacity,), dtype=np.float32)
        self.dones = np.zeros((capacity,), dtype=np.float32)
        self.ptr = 0
        self.size = 0

    def add(self, obs, action, reward, next_obs, done):
        self.obs[self.ptr] = obs
        self.next_obs[self.ptr] = next_obs
        self.actions[self.ptr] = action
        self.rewards[self.ptr] = reward
        self.dones[self.ptr] = float(done)
        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size, device):
        idx = np.random.randint(0, self.size, size=batch_size)
        return (
            torch.as_tensor(self.obs[idx], device=device),
            torch.as_tensor(self.actions[idx], device=device),
            torch.as_tensor(self.rewards[idx], device=device),
            torch.as_tensor(self.next_obs[idx], device=device),
            torch.as_tensor(self.dones[idx], device=device),
        )


def linear_epsilon(step, start=1.0, end=0.05, decay_steps=50_000):
    frac = min(1.0, step / decay_steps)
    return start + frac * (end - start)


def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    set_seed(args.seed)

    env = make_env(ENV_ID, seed=args.seed)
    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n

    q_net = QNetwork(obs_dim, n_actions).to(device)
    target_net = QNetwork(obs_dim, n_actions).to(device)
    target_net.load_state_dict(q_net.state_dict())
    use_target = not args.no_target_network

    optimizer = torch.optim.Adam(q_net.parameters(), lr=args.lr)
    buffer = ReplayBuffer(args.buffer_size, obs_dim)

    os.makedirs(args.out_dir, exist_ok=True)
    train_logger = CSVLogger(os.path.join(args.out_dir, "train_rewards.csv"),
                              ["env_step", "episode", "episode_return"])
    eval_logger = CSVLogger(os.path.join(args.out_dir, "eval_rewards.csv"),
                             ["env_step", "mean_return", "std_return"])

    obs, _ = env.reset(seed=args.seed)
    episode_return = 0.0
    episode_idx = 0
    next_eval_at = args.eval_every

    start_time = time.time()
    for step in range(1, args.total_steps + 1):
        epsilon = linear_epsilon(step, decay_steps=args.epsilon_decay_steps)
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                q_values = q_net(obs_t)
                action = int(torch.argmax(q_values, dim=1).item())

        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        buffer.add(obs, action, reward, next_obs, done)
        episode_return += reward
        obs = next_obs

        if done:
            train_logger.log({"env_step": step, "episode": episode_idx,
                               "episode_return": episode_return})
            episode_idx += 1
            obs, _ = env.reset()
            episode_return = 0.0

        # Learning update: one gradient step per env step, once warmup is done.
        if buffer.size >= args.learning_starts:
            s, a, r, s2, d = buffer.sample(args.batch_size, device)
            with torch.no_grad():
                # Ablation: if no target network, bootstrap directly off q_net.
                target_q_net = target_net if use_target else q_net
                next_q = target_q_net(s2).max(dim=1).values
                y = r + args.gamma * (1.0 - d) * next_q
            q = q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
            loss = F.smooth_l1_loss(q, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if use_target and step % args.target_update_freq == 0:
                target_net.load_state_dict(q_net.state_dict())

        if step >= next_eval_at:
            def greedy_action(o):
                with torch.no_grad():
                    o_t = torch.as_tensor(o, dtype=torch.float32, device=device).unsqueeze(0)
                    return int(torch.argmax(q_net(o_t), dim=1).item())

            mean_r, std_r = evaluate_policy(ENV_ID, greedy_action, n_episodes=args.eval_episodes,
                                             seed=args.seed, device=device)
            eval_logger.log({"env_step": step, "mean_return": mean_r, "std_return": std_r})
            tag = "no-target" if not use_target else "with-target"
            print(f"[DQN {tag}] step={step} eval_mean={mean_r:.1f} "
                  f"eval_std={std_r:.1f} epsilon={epsilon:.3f}")
            next_eval_at += args.eval_every

    env.close()
    print(f"Finished DQN run in {time.time() - start_time:.1f}s -> {args.out_dir}")


def build_arg_parser():
    p = argparse.ArgumentParser()
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--total-steps", type=int, default=200_000)
    p.add_argument("--eval-every", type=int, default=20_000)
    p.add_argument("--eval-episodes", type=int, default=20)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--buffer-size", type=int, default=50_000)
    p.add_argument("--batch-size", type=int, default=64)
    p.add_argument("--learning-starts", type=int, default=1_000)
    p.add_argument("--target-update-freq", type=int, default=500)
    p.add_argument("--epsilon-decay-steps", type=int, default=50_000)
    p.add_argument("--gamma", type=float, default=0.99)
    p.add_argument("--no-target-network", action="store_true",
                    help="Ablation: bootstrap directly off the online network "
                         "instead of a separate target network.")
    p.add_argument("--out-dir", type=str, default="results/dqn_normal")
    return p


if __name__ == "__main__":
    args = build_arg_parser().parse_args()
    train(args)


Writing dqn.py


In [3]:
%%writefile a2c.py
"""
A2C (Advantage Actor-Critic) on CartPole-v1, using synchronous vector environments.

Normal run:
    python a2c.py --seed 0 --out-dir results/a2c_normal

Ablation (no entropy bonus):
    python a2c.py --seed 0 --entropy-coef 0.0 --out-dir results/a2c_entropy0
"""
import argparse
import os
import time
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical

from common import set_seed, make_vector_env, CSVLogger, evaluate_policy

ENV_ID = "CartPole-v1"


class ActorCritic(nn.Module):
    """Shared MLP trunk (2 hidden layers, 128 units) with separate actor and critic heads."""
    def __init__(self, obs_dim, n_actions, hidden_size=128):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(obs_dim, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
        )
        self.actor = nn.Linear(hidden_size, n_actions)
        self.critic = nn.Linear(hidden_size, 1)

    def forward(self, obs):
        h = self.trunk(obs)
        logits = self.actor(h)
        value = self.critic(h).squeeze(-1)
        return logits, value


def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    set_seed(args.seed)

    n_envs = args.n_envs
    envs = make_vector_env(ENV_ID, n_envs=n_envs, seed=args.seed)
    obs_dim = envs.single_observation_space.shape[0]
    n_actions = envs.single_action_space.n

    net = ActorCritic(obs_dim, n_actions).to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=args.lr)

    os.makedirs(args.out_dir, exist_ok=True)
    train_logger = CSVLogger(os.path.join(args.out_dir, "train_rewards.csv"),
                              ["env_step", "episode", "episode_return"])
    eval_logger = CSVLogger(os.path.join(args.out_dir, "eval_rewards.csv"),
                             ["env_step", "mean_return", "std_return"])

    obs, _ = envs.reset(seed=args.seed)
    episode_returns = np.zeros(n_envs, dtype=np.float32)
    episode_idx = 0
    global_step = 0
    next_eval_at = args.eval_every

    start_time = time.time()
    while global_step < args.total_steps:
        act_buf, logp_buf, val_buf, rew_buf, done_buf, ent_buf = [], [], [], [], [], []

        for _ in range(args.rollout_length):
            obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device)
            logits, value = net(obs_t)
            dist = Categorical(logits=logits)
            action = dist.sample()

            next_obs, reward, terminated, truncated, infos = envs.step(action.cpu().numpy())
            done = np.logical_or(terminated, truncated)

            act_buf.append(action)
            logp_buf.append(dist.log_prob(action))
            val_buf.append(value)
            ent_buf.append(dist.entropy())
            rew_buf.append(torch.as_tensor(reward, dtype=torch.float32, device=device))
            done_buf.append(torch.as_tensor(done, dtype=torch.float32, device=device))

            episode_returns += reward
            for i, d in enumerate(done):
                if d:
                    train_logger.log({"env_step": global_step + args.rollout_length,
                                       "episode": episode_idx,
                                       "episode_return": float(episode_returns[i])})
                    episode_idx += 1
                    episode_returns[i] = 0.0

            obs = next_obs
            global_step += n_envs

        with torch.no_grad():
            next_obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device)
            _, next_value = net(next_obs_t)

        # Bootstrapped n-step returns, computed backward through the rollout.
        returns = [None] * args.rollout_length
        R = next_value
        for t in reversed(range(args.rollout_length)):
            R = rew_buf[t] + args.gamma * R * (1.0 - done_buf[t])
            returns[t] = R

        returns = torch.stack(returns)   # (T, n_envs)
        values = torch.stack(val_buf)    # (T, n_envs)
        logps = torch.stack(logp_buf)    # (T, n_envs)
        entropy = torch.stack(ent_buf).mean()
        advantages = returns - values

        policy_loss = -(logps * advantages.detach()).mean()
        value_loss = advantages.pow(2).mean()
        loss = policy_loss + args.value_coef * value_loss - args.entropy_coef * entropy

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), args.max_grad_norm)
        optimizer.step()

        if global_step >= next_eval_at:
            def greedy_action(o):
                with torch.no_grad():
                    o_t = torch.as_tensor(o, dtype=torch.float32, device=device).unsqueeze(0)
                    logits, _ = net(o_t)
                    return int(torch.argmax(logits, dim=1).item())

            mean_r, std_r = evaluate_policy(ENV_ID, greedy_action, n_episodes=args.eval_episodes,
                                             seed=args.seed, device=device)
            eval_logger.log({"env_step": global_step, "mean_return": mean_r, "std_return": std_r})
            print(f"[A2C entropy_coef={args.entropy_coef}] step={global_step} "
                  f"eval_mean={mean_r:.1f} eval_std={std_r:.1f}")
            next_eval_at += args.eval_every

    envs.close()
    print(f"Finished A2C run in {time.time() - start_time:.1f}s -> {args.out_dir}")


def build_arg_parser():
    p = argparse.ArgumentParser()
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--total-steps", type=int, default=200_000)
    p.add_argument("--eval-every", type=int, default=20_000)
    p.add_argument("--eval-episodes", type=int, default=20)
    p.add_argument("--lr", type=float, default=7e-4)
    p.add_argument("--rollout-length", type=int, default=5)
    p.add_argument("--n-envs", type=int, default=4)
    p.add_argument("--entropy-coef", type=float, default=0.01)
    p.add_argument("--value-coef", type=float, default=0.5)
    p.add_argument("--max-grad-norm", type=float, default=0.5)
    p.add_argument("--gamma", type=float, default=0.99)
    p.add_argument("--out-dir", type=str, default="results/a2c_normal")
    return p


if __name__ == "__main__":
    args = build_arg_parser().parse_args()
    train(args)


Writing a2c.py


In [4]:
%%writefile ppo.py
"""
PPO (Proximal Policy Optimization, clipped objective) on CartPole-v1.

Normal run (clip range 0.2):
    python ppo.py --seed 0 --clip-range 0.2 --out-dir results/ppo_clip02

Ablations:
    python ppo.py --seed 0 --clip-range 0.1 --out-dir results/ppo_clip01
    python ppo.py --seed 0 --clip-range 0.3 --out-dir results/ppo_clip03
"""
import argparse
import os
import time
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical

from common import set_seed, make_vector_env, CSVLogger, evaluate_policy

ENV_ID = "CartPole-v1"


class ActorCritic(nn.Module):
    """Shared MLP trunk (2 hidden layers, 128 units) with separate actor and critic heads."""
    def __init__(self, obs_dim, n_actions, hidden_size=128):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(obs_dim, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
        )
        self.actor = nn.Linear(hidden_size, n_actions)
        self.critic = nn.Linear(hidden_size, 1)

    def forward(self, obs):
        h = self.trunk(obs)
        return self.actor(h), self.critic(h).squeeze(-1)


def compute_gae(rewards, values, dones, next_value, gamma, lam):
    """
    rewards, values, dones: tensors of shape (T, n_envs)
    next_value: (n_envs,) bootstrap value for the state right after the rollout.
    Returns advantages, returns, both shape (T, n_envs).
    """
    T, n_envs = rewards.shape
    advantages = torch.zeros_like(rewards)
    last_gae = torch.zeros(n_envs, device=rewards.device)
    values_ext = torch.cat([values, next_value.unsqueeze(0)], dim=0)  # (T+1, n_envs)
    for t in reversed(range(T)):
        delta = rewards[t] + gamma * values_ext[t + 1] * (1.0 - dones[t]) - values_ext[t]
        last_gae = delta + gamma * lam * (1.0 - dones[t]) * last_gae
        advantages[t] = last_gae
    returns = advantages + values
    return advantages, returns


def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    set_seed(args.seed)

    n_envs = args.n_envs
    envs = make_vector_env(ENV_ID, n_envs=n_envs, seed=args.seed)
    obs_dim = envs.single_observation_space.shape[0]
    n_actions = envs.single_action_space.n

    net = ActorCritic(obs_dim, n_actions).to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=args.lr)

    os.makedirs(args.out_dir, exist_ok=True)
    train_logger = CSVLogger(os.path.join(args.out_dir, "train_rewards.csv"),
                              ["env_step", "episode", "episode_return"])
    eval_logger = CSVLogger(os.path.join(args.out_dir, "eval_rewards.csv"),
                             ["env_step", "mean_return", "std_return"])

    obs, _ = envs.reset(seed=args.seed)
    episode_returns = np.zeros(n_envs, dtype=np.float32)
    episode_idx = 0
    global_step = 0
    next_eval_at = args.eval_every

    start_time = time.time()
    while global_step < args.total_steps:
        obs_buf = torch.zeros(args.rollout_length, n_envs, obs_dim, device=device)
        act_buf = torch.zeros(args.rollout_length, n_envs, dtype=torch.long, device=device)
        logp_buf = torch.zeros(args.rollout_length, n_envs, device=device)
        val_buf = torch.zeros(args.rollout_length, n_envs, device=device)
        rew_buf = torch.zeros(args.rollout_length, n_envs, device=device)
        done_buf = torch.zeros(args.rollout_length, n_envs, device=device)

        for t in range(args.rollout_length):
            obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device)
            with torch.no_grad():
                logits, value = net(obs_t)
                dist = Categorical(logits=logits)
                action = dist.sample()
                logp = dist.log_prob(action)

            next_obs, reward, terminated, truncated, infos = envs.step(action.cpu().numpy())
            done = np.logical_or(terminated, truncated)

            obs_buf[t] = obs_t
            act_buf[t] = action
            logp_buf[t] = logp
            val_buf[t] = value
            rew_buf[t] = torch.as_tensor(reward, dtype=torch.float32, device=device)
            done_buf[t] = torch.as_tensor(done, dtype=torch.float32, device=device)

            episode_returns += reward
            for i, d in enumerate(done):
                if d:
                    train_logger.log({"env_step": global_step + (t + 1) * n_envs,
                                       "episode": episode_idx,
                                       "episode_return": float(episode_returns[i])})
                    episode_idx += 1
                    episode_returns[i] = 0.0

            obs = next_obs

        global_step += args.rollout_length * n_envs

        with torch.no_grad():
            next_obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device)
            _, next_value = net(next_obs_t)

        advantages, returns = compute_gae(rew_buf, val_buf, done_buf, next_value,
                                           args.gamma, args.gae_lambda)

        # Flatten (T, n_envs, ...) -> (T * n_envs, ...) for minibatch updates.
        b_obs = obs_buf.reshape(-1, obs_dim)
        b_actions = act_buf.reshape(-1)
        b_logp_old = logp_buf.reshape(-1)
        b_advantages = advantages.reshape(-1)
        b_returns = returns.reshape(-1)

        b_advantages = (b_advantages - b_advantages.mean()) / (b_advantages.std() + 1e-8)

        batch_size = b_obs.shape[0]
        idxs = np.arange(batch_size)
        for epoch in range(args.update_epochs):
            np.random.shuffle(idxs)
            for start in range(0, batch_size, args.minibatch_size):
                mb_idx = idxs[start:start + args.minibatch_size]
                mb_idx_t = torch.as_tensor(mb_idx, device=device)

                logits, values = net(b_obs[mb_idx_t])
                dist = Categorical(logits=logits)
                new_logp = dist.log_prob(b_actions[mb_idx_t])
                entropy = dist.entropy().mean()

                ratio = torch.exp(new_logp - b_logp_old[mb_idx_t])
                mb_adv = b_advantages[mb_idx_t]

                surr1 = ratio * mb_adv
                surr2 = torch.clamp(ratio, 1.0 - args.clip_range, 1.0 + args.clip_range) * mb_adv
                policy_loss = -torch.min(surr1, surr2).mean()

                value_loss = (values - b_returns[mb_idx_t]).pow(2).mean()

                loss = policy_loss + args.value_coef * value_loss - args.entropy_coef * entropy

                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), args.max_grad_norm)
                optimizer.step()

        if global_step >= next_eval_at:
            def greedy_action(o):
                with torch.no_grad():
                    o_t = torch.as_tensor(o, dtype=torch.float32, device=device).unsqueeze(0)
                    logits, _ = net(o_t)
                    return int(torch.argmax(logits, dim=1).item())

            mean_r, std_r = evaluate_policy(ENV_ID, greedy_action, n_episodes=args.eval_episodes,
                                             seed=args.seed, device=device)
            eval_logger.log({"env_step": global_step, "mean_return": mean_r, "std_return": std_r})
            print(f"[PPO clip={args.clip_range}] step={global_step} "
                  f"eval_mean={mean_r:.1f} eval_std={std_r:.1f}")
            next_eval_at += args.eval_every

    envs.close()
    print(f"Finished PPO run in {time.time() - start_time:.1f}s -> {args.out_dir}")


def build_arg_parser():
    p = argparse.ArgumentParser()
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--total-steps", type=int, default=200_000)
    p.add_argument("--eval-every", type=int, default=20_000)
    p.add_argument("--eval-episodes", type=int, default=20)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--rollout-length", type=int, default=128)
    p.add_argument("--n-envs", type=int, default=4)
    p.add_argument("--gae-lambda", type=float, default=0.95)
    p.add_argument("--clip-range", type=float, default=0.2)
    p.add_argument("--update-epochs", type=int, default=4)
    p.add_argument("--minibatch-size", type=int, default=128)
    p.add_argument("--entropy-coef", type=float, default=0.01)
    p.add_argument("--value-coef", type=float, default=0.5)
    p.add_argument("--max-grad-norm", type=float, default=0.5)
    p.add_argument("--gamma", type=float, default=0.99)
    p.add_argument("--out-dir", type=str, default="results/ppo_clip02")
    return p


if __name__ == "__main__":
    args = build_arg_parser().parse_args()
    train(args)


Writing ppo.py


In [5]:
# 1. Run DQN Normal and Ablation
!python dqn.py --seed 0 --out-dir results/dqn_normal
!python dqn.py --seed 0 --no-target-network --out-dir results/dqn_no_target

# 2. Run A2C Normal and Ablation
!python a2c.py --seed 0 --out-dir results/a2c_normal
!python a2c.py --seed 0 --entropy-coef 0.0 --out-dir results/a2c_entropy0

# 3. Run PPO Normal and Ablations
!python ppo.py --seed 0 --clip-range 0.2 --out-dir results/ppo_clip02
!python ppo.py --seed 0 --clip-range 0.1 --out-dir results/ppo_clip01
!python ppo.py --seed 0 --clip-range 0.3 --out-dir results/ppo_clip03

[DQN with-target] step=20000 eval_mean=87.2 eval_std=5.3 epsilon=0.620
[DQN with-target] step=40000 eval_mean=208.4 eval_std=27.1 epsilon=0.240
[DQN with-target] step=60000 eval_mean=89.0 eval_std=1.8 epsilon=0.050
[DQN with-target] step=80000 eval_mean=89.0 eval_std=1.6 epsilon=0.050
[DQN with-target] step=100000 eval_mean=82.4 eval_std=9.3 epsilon=0.050
[DQN with-target] step=120000 eval_mean=104.5 eval_std=5.8 epsilon=0.050
[DQN with-target] step=140000 eval_mean=89.5 eval_std=1.8 epsilon=0.050
[DQN with-target] step=160000 eval_mean=87.9 eval_std=1.8 epsilon=0.050
[DQN with-target] step=180000 eval_mean=81.0 eval_std=15.4 epsilon=0.050
[DQN with-target] step=200000 eval_mean=500.0 eval_std=0.0 epsilon=0.050
Finished DQN run in 430.9s -> results/dqn_normal
[DQN no-target] step=20000 eval_mean=9.4 eval_std=0.6 epsilon=0.620
[DQN no-target] step=40000 eval_mean=9.4 eval_std=0.6 epsilon=0.240
[DQN no-target] step=60000 eval_mean=9.4 eval_std=0.6 epsilon=0.050
[DQN no-target] step=80000

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Set a clean style for the plots
plt.style.use('seaborn-v0_8-whitegrid')

# Helper function to plot evaluation curves with shaded standard deviation
def plot_eval_curve(ax, csv_path, label, color):
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        ax.plot(df['env_step'], df['mean_return'], label=label, color=color, linewidth=2)
        ax.fill_between(df['env_step'], 
                        df['mean_return'] - df['std_return'], 
                        df['mean_return'] + df['std_return'], 
                        color=color, alpha=0.2)

# Helper function to smooth and plot training curves
def plot_train_curve(ax, csv_path, label, color, window=100):
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        # Smoothing the training data because RL returns are very noisy
        smoothed = df['episode_return'].rolling(window=window, min_periods=1).mean()
        ax.plot(df['env_step'], smoothed, label=label, color=color, alpha=0.8)

print("Generating plots...")

# ==========================================
# EXPERIMENT 1: DQN Plots (Train & Eval)
# ==========================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Training Curve
plot_train_curve(ax1, 'results/dqn_normal/train_rewards.csv', 'DQN Normal', 'blue')
plot_train_curve(ax1, 'results/dqn_no_target/train_rewards.csv', 'DQN No Target', 'red')
ax1.set_title('DQN: Training Reward Curve (Smoothed)')
ax1.set_xlabel('Environment Steps')
ax1.set_ylabel('Episode Return')
ax1.legend()

# Eval Curve
plot_eval_curve(ax2, 'results/dqn_normal/eval_rewards.csv', 'DQN Normal', 'blue')
plot_eval_curve(ax2, 'results/dqn_no_target/eval_rewards.csv', 'DQN No Target', 'red')
ax2.set_title('DQN: Evaluation Reward Curve')
ax2.set_xlabel('Environment Steps')
ax2.set_ylabel('Mean Return (20 episodes)')
ax2.legend()

plt.tight_layout()
plt.savefig('dqn_plots.png', dpi=300)
plt.close()

# ==========================================
# EXPERIMENT 2: A2C Plot
# ==========================================
fig, ax = plt.subplots(figsize=(8, 5))
plot_eval_curve(ax, 'results/a2c_normal/eval_rewards.csv', 'A2C Normal (Entropy=0.01)', 'green')
plot_eval_curve(ax, 'results/a2c_entropy0/eval_rewards.csv', 'A2C Ablation (Entropy=0)', 'orange')
ax.set_title('A2C: Evaluation Reward Curve')
ax.set_xlabel('Environment Steps')
ax.set_ylabel('Mean Return')
ax.legend()
plt.tight_layout()
plt.savefig('a2c_plot.png', dpi=300)
plt.close()

# ==========================================
# EXPERIMENT 3: PPO Plot
# ==========================================
fig, ax = plt.subplots(figsize=(8, 5))
plot_eval_curve(ax, 'results/ppo_clip01/eval_rewards.csv', 'PPO Clip 0.1', 'purple')
plot_eval_curve(ax, 'results/ppo_clip02/eval_rewards.csv', 'PPO Normal (Clip 0.2)', 'blue')
plot_eval_curve(ax, 'results/ppo_clip03/eval_rewards.csv', 'PPO Clip 0.3', 'red')
ax.set_title('PPO: Evaluation Reward Curve (Clipping Ablation)')
ax.set_xlabel('Environment Steps')
ax.set_ylabel('Mean Return')
ax.legend()
plt.tight_layout()
plt.savefig('ppo_plot.png', dpi=300)
plt.close()

# ==========================================
# EXPERIMENT 4: Final Comparison Plot
# ==========================================
fig, ax = plt.subplots(figsize=(8, 5))
plot_eval_curve(ax, 'results/dqn_normal/eval_rewards.csv', 'DQN', 'blue')
plot_eval_curve(ax, 'results/a2c_normal/eval_rewards.csv', 'A2C', 'green')
plot_eval_curve(ax, 'results/ppo_clip02/eval_rewards.csv', 'PPO', 'red')
ax.set_title('Final Comparison: DQN vs A2C vs PPO')
ax.set_xlabel('Environment Steps')
ax.set_ylabel('Mean Return')
ax.legend()
plt.tight_layout()
plt.savefig('final_comparison_plot.png', dpi=300)
plt.close()

print("All plots saved as PNG files!")

# ==========================================
# GENERATE FINAL EVALUATION TABLE
# ==========================================
print("\n" + "="*50)
print("FINAL EVALUATION TABLE (At 200,000 steps)")
print("="*50)
print(f"{'Algorithm':<15} | {'Mean Return':<15} | {'Std Dev':<15}")
print("-" * 50)

def print_final_score(path, name):
    if os.path.exists(path):
        df = pd.read_csv(path)
        # Get the very last row
        final_row = df.iloc[-1]
        print(f"{name:<15} | {final_row['mean_return']:<15.1f} | {final_row['std_return']:<15.1f}")
    else:
        print(f"{name:<15} | Missing Data    | Missing Data")

print_final_score('results/dqn_normal/eval_rewards.csv', 'DQN')
print_final_score('results/a2c_normal/eval_rewards.csv', 'A2C')
print_final_score('results/ppo_clip02/eval_rewards.csv', 'PPO')
print("="*50 + "\n")

Generating plots...
All plots saved as PNG files!

FINAL EVALUATION TABLE (At 200,000 steps)
Algorithm       | Mean Return     | Std Dev        
--------------------------------------------------
DQN             | 500.0           | 0.0            
A2C             | 76.7            | 4.6            
PPO             | 194.3           | 9.4            

